In [64]:
import pandas as pd
import glob
import os


In [65]:

# TASK 1: Load and Inspect the Dataset

print("TASK 1: Load and Inspect the Dataset")

DATA_PATH = r"D:\Yoobee College\DA\Activities\MSE-803-Data-Analytics\week 2\beijing+multi+site+air+quality+data\PRSA2017_Data_20130301-20170228"

# Load ALL CSV files and combine into one DataFrame
all_files = glob.glob(os.path.join(DATA_PATH, "*.csv"))
#print(f"\nFound {len(all_files)} CSV files:")
for f in all_files:
  os.path.basename(f)

df_list = []
for file in all_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)
#print(f"\n All files combined into one DataFrame.")

# Display first 5 rows
print("\n First 5 Rows ")
print(df.head())

# Column names and data types
print("\n Column Names and Data Types ")
print(df.dtypes)

# Total rows and columns
print(f"\n Dataset Size ")
print(f"Total Rows   : {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

TASK 1: Load and Inspect the Dataset

 First 5 Rows 
   No  year  month  day  hour  PM2.5  PM10   SO2   NO2     CO    O3  TEMP  \
0   1  2013      3    1     0    4.0   4.0   4.0   7.0  300.0  77.0  -0.7   
1   2  2013      3    1     1    8.0   8.0   4.0   7.0  300.0  77.0  -1.1   
2   3  2013      3    1     2    7.0   7.0   5.0  10.0  300.0  73.0  -1.1   
3   4  2013      3    1     3    6.0   6.0  11.0  11.0  300.0  72.0  -1.4   
4   5  2013      3    1     4    3.0   3.0  12.0  12.0  300.0  72.0  -2.0   

     PRES  DEWP  RAIN   wd  WSPM       station  
0  1023.0 -18.8   0.0  NNW   4.4  Aotizhongxin  
1  1023.2 -18.2   0.0    N   4.7  Aotizhongxin  
2  1023.5 -18.2   0.0  NNW   5.6  Aotizhongxin  
3  1024.5 -19.4   0.0   NW   3.1  Aotizhongxin  
4  1025.2 -19.5   0.0    N   2.0  Aotizhongxin  

 Column Names and Data Types 
No           int64
year         int64
month        int64
day          int64
hour         int64
PM2.5      float64
PM10       float64
SO2        float64
NO2    

In [67]:

# TASK 2: Data Cleaning

print("TASK 2: Data Cleaning")


# Step 1: Identify missing values
print("\n Missing Values (Count and Percentage) ")
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"Missing Count": missing_count, "Missing %": missing_pct})
print(missing_summary[missing_summary["Missing Count"] > 0])

# Step 2: Create datetime column
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
print("\n Datetime column created.")

# Step 3: Sort by station and datetime
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)
print(" Sorted by station and datetime.")

# Step 4: Forward-fill then backfill within each station group
numeric_cols = df.select_dtypes(include="number").columns.tolist()
df[numeric_cols] = df.groupby("station")[numeric_cols].transform(
    lambda x: x.ffill().bfill()
)
print(" Missing values filled using forward-fill and backfill per station.")

# Step 5: Remove rows still missing pollutant values + duplicates
pollutants = [c for c in ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"] if c in df.columns]
before = len(df)
df.dropna(subset=pollutants, inplace=True)
df.drop_duplicates(inplace=True)
after = len(df)
print(f" Removed {before - after} invalid/duplicate rows.")

print(f"\nFinal dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())


 

TASK 2: Data Cleaning

 Missing Values (Count and Percentage) 
    Missing Count  Missing %
wd           1822       0.43

 Datetime column created.
 Sorted by station and datetime.
 Missing values filled using forward-fill and backfill per station.
 Removed 0 invalid/duplicate rows.

Final dataset: 420768 rows, 19 columns
   No  year  month  day  hour  PM2.5  PM10   SO2   NO2     CO    O3  TEMP  \
0   1  2013      3    1     0    4.0   4.0   4.0   7.0  300.0  77.0  -0.7   
1   2  2013      3    1     1    8.0   8.0   4.0   7.0  300.0  77.0  -1.1   
2   3  2013      3    1     2    7.0   7.0   5.0  10.0  300.0  73.0  -1.1   
3   4  2013      3    1     3    6.0   6.0  11.0  11.0  300.0  72.0  -1.4   
4   5  2013      3    1     4    3.0   3.0  12.0  12.0  300.0  72.0  -2.0   

     PRES  DEWP  RAIN   wd  WSPM       station            datetime  
0  1023.0 -18.8   0.0  NNW   4.4  Aotizhongxin 2013-03-01 00:00:00  
1  1023.2 -18.2   0.0    N   4.7  Aotizhongxin 2013-03-01 01:00:00  
2  102

In [69]:


# TASK 3: Basic Statistical Analysis

print("TASK 3: Basic Statistical Analysis")


# Step 1: Overall descriptive statistics
print("\n Descriptive Statistics (All Stations) ")
stats = df[pollutants + ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]].agg(
    ["mean", "median", "min", "max", "std"]
).round(2)
print(stats.to_string())

# Step 2: PM2.5 stats grouped by station
if "PM2.5" in df.columns and "station" in df.columns:
    print("\n PM2.5 Statistics by Station ")
    pm25_by_station = df.groupby("station")["PM2.5"].agg(
        Mean="mean", Median="median", Min="min", Max="max", Std="std"
    ).round(2)
    print(pm25_by_station.to_string())



TASK 3: Basic Statistical Analysis

 Descriptive Statistics (All Stations) 
         PM2.5    PM10     SO2     NO2        CO       O3   TEMP     PRES   DEWP   RAIN   WSPM
mean     80.15  105.06   15.93   50.53   1240.24    57.44  13.53  1010.75   2.48   0.06   1.73
median   55.00   82.00    7.00   43.00    900.00    44.00  14.50  1010.40   3.00   0.00   1.40
min       2.00    2.00    0.29    1.03    100.00     0.21 -19.90   982.40 -43.40   0.00   0.00
max     999.00  999.00  500.00  290.00  10000.00  1071.00  41.60  1042.80  29.10  72.50  13.20
std      81.30   92.67   22.28   35.31   1170.86    58.31  11.44    10.47  13.80   0.82   1.25

 PM2.5 Statistics by Station 
                Mean  Median  Min    Max    Std
station                                        
Aotizhongxin   83.16    60.0  3.0  898.0  82.29
Changping      71.12    47.0  2.0  882.0  72.42
Dingling       67.43    41.0  3.0  881.0  74.82
Dongsi         86.31    61.0  3.0  737.0  86.34
Guanyuan       83.05    59.0  2.0  